# 🚲 Operationalizing ML Across Clouds with Delta Sharing

This demo demonstrates how to train and register models in Unity Catalog from one workspace (Dev) and then use the shared artifacts in another workspace (Prod) for A/B testing and model inferencing.

Databricks provides a hosted MLflow Model Registry in Unity Catalog, fully compatible with the open-source MLflow Python client. 
 
Key benefits include:
- Centralized access control
- Full auditing and lineage tracking
- Cross-workspace model discovery and collaboration

**Dataset:** [Bike Sharing Dataset](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

---

## 📚 Workflow Overview

- Load the Bike Sharing dataset.
- Train and register multiple models to Unity Catalog (UC)
- Share the models via Delta Sharing to another workspace.
- Load the shared models in a second workspace.
- Perform A/B testing between the challenger and champion models.

---

## ⚙️ Requirements

- Databricks Runtime: **16.x ML** or later
- Unity Catalog enabled in both workspaces
- AutoML enabled

---

## Workspace A (Train + Register)

1. Trigger an AutoML experiment.
2. Log the best model and experiment runs with MLflow.
3. Register the best model to UC Model Registry as:  
   `alexander_booth.default.bike_sharing_uc_model`
4. Share the registered models via Delta Sharing to Workspace B.

---

## Workspace B (Consume + Predict)

5. Accept the shared models via Unity Catalog UI or API.
6. Access the shared models using:  
   `models:/shared_catalog.shared_schema.bike_sharing_model`
7. Load the model and run batch predictions on new data.

---


## 🗂️ Unity Catalog Setup

Set the catalog and schema where the model will be registered.

You will need the following privileges:
- `USE CATALOG` on the target catalog
- `USE SCHEMA` and `CREATE MODEL` on the target schema

Update the catalog and schema below if necessary before proceeding.


In [0]:
# Catalog name where the model artifacts will be stored in Unity Catalog
CATALOG_NAME = "alexander_booth" 
SCHEMA_NAME = "default" # Schema (database) name within the catalog to organize model artifacts
TABLE_NAME = "bike_sharing_training_data"
MODEL_NAME = "bike_sharing_uc_model"

## 📊 Data Description

Dataset: [Bike Sharing Dataset (UCI ML Repository)](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset)

---

### 🔹 Features

| Column      | Description |
|:------------|:-------------|
| dteday      | Date |
| season      | Season (1: Spring, 2: Summer, 3: Fall, 4: Winter) |
| yr          | Year (0: 2011, 1: 2012) |
| mnth        | Month (1–12) |
| hr          | Hour of day (0–23) |
| holiday     | 1 if holiday, 0 otherwise |
| weekday     | Day of week (0 = Sunday) |
| workingday  | 1 if working day, 0 if weekend/holiday |
| weathersit  | Weather condition (1–4 scale) |
| temp        | Normalized temperature |
| atemp       | Normalized "feels like" temperature |
| hum         | Normalized humidity |
| windspeed   | Normalized wind speed |

---

### 🔹 Labels

| Column      | Description |
|:------------|:-------------|
| casual      | Count of casual users |
| registered  | Count of registered users |
| cnt         | Total rental count (casual + registered) |


In [0]:
# Load the bike-sharing dataset from the specified CSV file into a Spark DataFrame.
# The dataset includes hourly bike rental data. The `header=True` option indicates that the first row contains column names,
# and `inferSchema=True` allows Spark to automatically infer data types for each column
bike_df = spark.read.csv(
    "/databricks-datasets/bikeSharing/data-001/hour.csv",
    header=True,
    inferSchema=True
)

# Print the schema of the loaded DataFrame to understand the structure and data types of its columns
bike_df.printSchema()

root
 |-- instant: integer (nullable = true)
 |-- dteday: date (nullable = true)
 |-- season: integer (nullable = true)
 |-- yr: integer (nullable = true)
 |-- mnth: integer (nullable = true)
 |-- hr: integer (nullable = true)
 |-- holiday: integer (nullable = true)
 |-- weekday: integer (nullable = true)
 |-- workingday: integer (nullable = true)
 |-- weathersit: integer (nullable = true)
 |-- temp: double (nullable = true)
 |-- atemp: double (nullable = true)
 |-- hum: double (nullable = true)
 |-- windspeed: double (nullable = true)
 |-- casual: integer (nullable = true)
 |-- registered: integer (nullable = true)
 |-- cnt: integer (nullable = true)



In [0]:
# Step 2: Prepare data for training - Convert Spark DataFrame to Pandas DataFrame.

# Select only the relevant columns from the dataset that will be used for modeling
# These columns include features such as season, year, month, hour, weather conditions, and target variable (`cnt`)
selected_columns = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 
                    'workingday', 'weathersit', 'temp', 'atemp', 'hum', 
                    'windspeed', 'cnt']

# Filter the DataFrame to include only the selected columns
bike_df = bike_df.select(selected_columns)
display(bike_df.head(5))

season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,0,1,1,0,6,0,1,0.22,0.2727,0.8,0.0,40
1,0,1,2,0,6,0,1,0.22,0.2727,0.8,0.0,32
1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


In [0]:
# Save it as a table
bike_df.write.mode("overwrite").saveAsTable(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}")

## ✅ Next Steps

Now that the data is ready to go, the next steps include:

1. **Train the models using AutoML**  

2. **Register the models to Unity Catalog**  

Let's move to the *01-automl-training* notebook.